In [1]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import (
    LongformerTokenizer, LongformerForSequenceClassification,
    BertTokenizer, BertForSequenceClassification,
    AutoTokenizer, AutoModelForSequenceClassification
)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, precision_score, recall_score

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ==== BERT/CLINICALBERT DATASET (CHUNKED) ====
class ChunkedTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, chunk_size=510, max_length=512, doc_max_length=4096):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.chunk_size = chunk_size
        self.max_length = max_length
        self.doc_max_length = doc_max_length

    def __len__(self):
        return len(self.texts)

    def chunk_text(self, text):
        tokens = self.tokenizer.encode(text, add_special_tokens=False)
        tokens = tokens[:self.doc_max_length]
        chunks = []
        for i in range(0, len(tokens), self.chunk_size):
            chunk = tokens[i:i+self.chunk_size]
            chunk = [self.tokenizer.cls_token_id] + chunk + [self.tokenizer.sep_token_id]
            if len(chunk) < self.max_length:
                chunk += [self.tokenizer.pad_token_id] * (self.max_length - len(chunk))
            chunk = chunk[:self.max_length]
            chunks.append(chunk)
        return chunks

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = float(self.labels[idx])
        chunks = self.chunk_text(text)
        return {
            'chunks': torch.tensor(chunks, dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.float),
            'num_chunks': len(chunks)
        }

def bert_collate_fn(batch):
    all_chunks = [item['chunks'] for item in batch]
    all_labels = torch.tensor([item['label'] for item in batch], dtype=torch.float)
    all_num_chunks = [item['num_chunks'] for item in batch]
    flat_chunks = torch.cat(all_chunks, dim=0)
    return {
        'chunks': flat_chunks,
        'labels': all_labels,
        'num_chunks': all_num_chunks
    }

def get_bert_predictions(model, data_loader, device, tokenizer, pooling='max'):
    model.eval()
    all_preds, all_probs = [], []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Predicting"):
            chunks = batch['chunks'].to(device)
            labels = batch['labels'].to(device)
            num_chunks = batch['num_chunks']
            outputs = model(input_ids=chunks, attention_mask=(chunks != tokenizer.pad_token_id))
            logits = outputs.logits.view(-1)
            chunk_idx = 0
            pooled_probs = []
            for nc in num_chunks:
                chunk_logits = logits[chunk_idx:chunk_idx+nc]
                prob = torch.sigmoid(chunk_logits)
                if pooling == 'max':
                    pooled_prob = torch.max(prob)
                elif pooling == 'mean':
                    pooled_prob = torch.mean(prob)
                else:
                    raise ValueError(f"Unknown pooling: {pooling}")
                pooled_probs.append(pooled_prob.item())
                chunk_idx += nc
            all_probs.extend(pooled_probs)
            all_preds.extend([int(p >= 0.5) for p in pooled_probs])
    return np.array(all_preds), np.array(all_probs)

In [4]:
# --- Model configs: ONLY BERT and CLINICALBERT ---
model_configs = [
    {
        "name": "BERT_Hate_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/bert_hate_mimic_0522",
        "tokenizer_cls": BertTokenizer,
        "model_cls": BertForSequenceClassification,
        "max_length": 512
    },
    {
        "name": "BERT_Hate_Phenotype_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/bert_hate_phenotype_mimic_0523",
        "tokenizer_cls": BertTokenizer,
        "model_cls": BertForSequenceClassification,
        "max_length": 512
    },
    {
        "name": "BERT_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/mimic_bert_0522",
        "tokenizer_cls": BertTokenizer,
        "model_cls": BertForSequenceClassification,
        "max_length": 512
    },
    {
        "name": "ClinicalBERT_Hate_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/ClinicalBERT_hate_mimic_0523",
        "tokenizer_cls": AutoTokenizer,
        "model_cls": AutoModelForSequenceClassification,
        "max_length": 512
    },
    {
        "name": "ClinicalBERT_Hate_Phenotype_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/ClinicalBERT_hate_phenotype_mimic_0524",
        "tokenizer_cls": AutoTokenizer,
        "model_cls": AutoModelForSequenceClassification,
        "max_length": 512
    },
    {
        "name": "ClinicalBERT_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/clinicalbert_mimic_0523",
        "tokenizer_cls": AutoTokenizer,
        "model_cls": AutoModelForSequenceClassification,
        "max_length": 512
    },
]


In [5]:
mimic_test = pd.read_csv('/content/drive/My Drive/EHR_PROJ/DATA/mimic_no_preprocess_binary_val.csv')
test_texts = mimic_test['text'].tolist()
test_labels = mimic_test['label'].tolist()

In [6]:
BATCH_SIZE = 48  # or lower if out of memory
device = 'cuda' if torch.cuda.is_available() else 'cpu'

all_results = []
summary_results = []

for cfg in model_configs:
    print(f"\nEvaluating model: {cfg['name']}")
    tokenizer = cfg['tokenizer_cls'].from_pretrained(cfg['model_dir'])
    model = cfg['model_cls'].from_pretrained(cfg['model_dir'])
    model.to(device)
    dataset = ChunkedTextDataset(
        test_texts, test_labels, tokenizer,
        chunk_size=510, max_length=cfg['max_length'], doc_max_length=4096
    )
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=bert_collate_fn)

    preds, probs = get_bert_predictions(model, loader, device, tokenizer, pooling='max')
    for idx, (pred, prob, true_label) in enumerate(zip(preds, probs, test_labels)):
        all_results.append({
            "model": cfg["name"],
            "doc_idx": idx,
            "true_label": true_label,
            "pred_label": pred,
            "prob": prob
        })

    # Calculate metrics for the whole test set
    acc = accuracy_score(test_labels, preds)
    f1 = f1_score(test_labels, preds)
    prec = precision_score(test_labels, preds)
    rec = recall_score(test_labels, preds)
    try:
        auc = roc_auc_score(test_labels, probs)
    except Exception:
        auc = np.nan

    summary_results.append({
        "model": cfg["name"],
        "accuracy": acc,
        "f1": f1,
        "precision": prec,
        "recall": rec,
        "auc": auc
    })

    print(f"Accuracy: {acc:.4f} | F1: {f1:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | AUC: {auc:.4f}")
    print(classification_report(test_labels, preds, digits=4))


Evaluating model: BERT_Hate_MIMIC


Predicting: 100%|██████████| 18/18 [01:02<00:00,  3.49s/it]


Accuracy: 0.8063 | F1: 0.8710 | Precision: 0.8451 | Recall: 0.8985 | AUC: 0.7682
              precision    recall  f1-score   support

           0     0.6738    0.5600    0.6117       225
           1     0.8451    0.8985    0.8710       601

    accuracy                         0.8063       826
   macro avg     0.7594    0.7293    0.7413       826
weighted avg     0.7984    0.8063    0.8003       826


Evaluating model: BERT_Hate_Phenotype_MIMIC


Predicting: 100%|██████████| 18/18 [01:02<00:00,  3.44s/it]


Accuracy: 0.8390 | F1: 0.8921 | Precision: 0.8703 | Recall: 0.9151 | AUC: 0.8293
              precision    recall  f1-score   support

           0     0.7371    0.6356    0.6826       225
           1     0.8703    0.9151    0.8921       601

    accuracy                         0.8390       826
   macro avg     0.8037    0.7753    0.7874       826
weighted avg     0.8340    0.8390    0.8351       826


Evaluating model: BERT_MIMIC


...

Accuracy: 0.8123 | F1: 0.8707 | Precision: 0.8729 | Recall: 0.8686 | AUC: 0.8234
              precision    recall  f1-score   support

           0     0.6535    0.6622    0.6578       225
           1     0.8729    0.8686    0.8707       601

    accuracy                         0.8123       826
   macro avg     0.7632    0.7654    0.7643       826
weighted avg     0.8131    0.8123    0.8127       826


Evaluating model: ClinicalBERT_Hate_Phenotype_MIMIC


Predicting: 100%|██████████| 18/18 [00:38<00:00,  2.14s/it]


Accuracy: 0.8354 | F1: 0.8901 | Precision: 0.8650 | Recall: 0.9168 | AUC: 0.8286
              precision    recall  f1-score   support

           0     0.7354    0.6178    0.6715       225
           1     0.8650    0.9168    0.8901       601

    accuracy                         0.8354       826
   macro avg     0.8002    0.7673    0.7808       826
weighted avg     0.8297    0.8354    0.8306       826


Evaluating model: ClinicalBERT_MIMIC


Predicting: 100%|██████████| 18/18 [00:38<00:00,  2.14s/it]

Accuracy: 0.8002 | F1: 0.8677 | Precision: 0.8375 | Recall: 0.9002 | AUC: 0.7726
              precision    recall  f1-score   support

           0     0.6667    0.5333    0.5926       225
           1     0.8375    0.9002    0.8677       601

    accuracy                         0.8002       826
   macro avg     0.7521    0.7167    0.7301       826
weighted avg     0.7909    0.8002    0.7927       826



In [7]:
results_df = pd.DataFrame(all_results)
results_df

,model,doc_idx,true_label,pred_label,prob
0,BERT_Hate_MIMIC,0,1,1,0.999464
1,BERT_Hate_MIMIC,1,1,1,0.999556
2,BERT_Hate_MIMIC,2,0,0,0.000536
3,BERT_Hate_MIMIC,3,0,0,0.000970
4,BERT_Hate_MIMIC,4,1,1,0.907820
...,...,...,...,...,...
4951,ClinicalBERT_MIMIC,821,1,1,0.999768
4952,ClinicalBERT_MIMIC,822,1,1,0.999766
4953,ClinicalBERT_MIMIC,823,1,1,0.999770
4954,ClinicalBERT_MIMIC,824,0,0,0.000373


In [8]:
summary_df = pd.DataFrame(summary_results)
summary_df

,model,accuracy,f1,precision,recall,auc
0,BERT_Hate_MIMIC,0.806295,0.870968,0.845070,0.898502,0.768238
1,BERT_Hate_Phenotype_MIMIC,0.838983,0.892133,0.870253,0.915141,0.829310
2,BERT_MIMIC,0.757869,0.822380,0.881905,0.770383,0.770146
3,ClinicalBERT_Hate_MIMIC,0.812349,0.870726,0.872910,0.868552,0.823357
4,ClinicalBERT_Hate_Phenotype_MIMIC,0.835351,0.890145,0.864992,0.916805,0.828564
5,ClinicalBERT_MIMIC,0.800242,0.867682,0.837461,0.900166,0.772646


In [11]:
# Pivot predicted labels
pred_pivot = results_df.pivot(index="doc_idx", columns="model", values="pred_label")
pred_pivot.columns = [f"{col}_pred" for col in pred_pivot.columns]

# True labels (from the first occurrence per doc_idx)
true_labels = results_df.drop_duplicates("doc_idx").sort_values("doc_idx")["true_label"].values

# Assemble into wide-format DataFrame
wide_preds = pd.DataFrame({
    "true_label": true_labels
}, index=pred_pivot.index)

wide_preds = pd.concat([wide_preds, pred_pivot], axis=1)

In [12]:
wide_preds

,true_label,BERT_Hate_MIMIC_pred,BERT_Hate_Phenotype_MIMIC_pred,BERT_MIMIC_pred,ClinicalBERT_Hate_MIMIC_pred,ClinicalBERT_Hate_Phenotype_MIMIC_pred,ClinicalBERT_MIMIC_pred
doc_idx,,,,,,,
0,1,1,1,1,1,1,1
1,1,1,1,1,1,1,1
2,0,0,0,0,0,0,0
3,0,0,0,0,0,0,1
4,1,1,1,0,1,1,1
...,...,...,...,...,...,...,...
821,1,1,1,0,1,1,1
822,1,1,1,1,1,1,1
823,1,1,1,1,1,1,1


In [13]:
model_pred_cols = [col for col in wide_preds.columns if col.endswith('_pred')]

# Calculate accuracy for each model
accuracies = {}
for col in model_pred_cols:
    acc = (wide_preds[col] == wide_preds['true_label']).mean()
    # Remove the '_pred' suffix for cleaner model name
    accuracies[col.replace('_pred', '')] = acc

# Display as DataFrame
acc_df = pd.DataFrame(list(accuracies.items()), columns=['model', 'accuracy'])
print(acc_df)

                               model  accuracy
0                    BERT_Hate_MIMIC  0.806295
1          BERT_Hate_Phenotype_MIMIC  0.838983
2                         BERT_MIMIC  0.757869
3            ClinicalBERT_Hate_MIMIC  0.812349
4  ClinicalBERT_Hate_Phenotype_MIMIC  0.835351
5                 ClinicalBERT_MIMIC  0.800242


In [14]:
wide_preds.to_csv("/content/drive/My Drive/EHR_PROJ/Results/all_bert_clinicalbert_wide_preds.csv")